# 14 — DLA Parity Sectors on IBM Quantum Hardware

Simulator result (notebook colab/kaggle_dla_parity_decoherence):
Odd (bottom-up) sector is MORE ROBUST to decoherence than even (top-down).
t=-18.6, p<0.0001 across 20 seeds.

**This notebook runs the same experiment on ibm_fez (156 qubits).**

Binary outcome:
- Asymmetry survives real noise → Physical Review Letters territory
- Asymmetry vanishes → simulator artefact of symmetric noise models

In [ ]:
import json

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator

from scpn_quantum_control.bridge import OMEGA_N_16, build_knm_paper27, knm_to_ansatz
from scpn_quantum_control.phase.phase_vqe import PhaseVQE

In [ ]:
N = 4
K = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

# Get ground state via VQE
vqe = PhaseVQE(K, omega, ansatz_reps=2)
sol = vqe.solve(maxiter=200, seed=0)
print(f"Ground energy: {sol['ground_energy']:.6f}")

ansatz = knm_to_ansatz(K, reps=2)
bound = ansatz.assign_parameters(sol["optimal_params"])
sv_ground = Statevector.from_instruction(bound)

# Parity operator P = Z^N
P_label = "Z" * N
P = SparsePauliOp(P_label)
parity_ev = float(sv_ground.expectation_value(P).real)
print(f"Ground state parity <P>: {parity_ev:.4f}")

# Project into even and odd sectors
dim = 2**N
P_matrix = P.to_matrix()
sv_vec = sv_ground.data

P_even_proj = (np.eye(dim) + P_matrix) / 2
P_odd_proj = (np.eye(dim) - P_matrix) / 2

sv_even = P_even_proj @ sv_vec
sv_odd = P_odd_proj @ sv_vec

norm_even = np.linalg.norm(sv_even)
norm_odd = np.linalg.norm(sv_odd)
print(f"Even weight: {norm_even**2:.4f}, Odd weight: {norm_odd**2:.4f}")

if norm_even > 1e-10:
    sv_even = sv_even / norm_even
if norm_odd > 1e-10:
    sv_odd = sv_odd / norm_odd

## Build Circuits for Even and Odd Sector States

In [ ]:
def build_parity_circuit(state_vector, N, n_noise_layers=5):
    """Circuit that prepares a parity state then exposes it to noise."""
    qc = QuantumCircuit(N)
    qc.initialize(state_vector)
    # Near-identity layers that accumulate noise on real hardware
    for _ in range(n_noise_layers):
        for i in range(N):
            qc.rz(0.01, i)
        for i in range(N - 1):
            qc.cx(i, i + 1)
            qc.cx(i, i + 1)  # CX^2 = I (but accumulates noise)
    qc.measure_all()
    return qc


qc_even = build_parity_circuit(sv_even, N)
qc_odd = build_parity_circuit(sv_odd, N)

print(f"Even circuit: {qc_even.depth()} depth, {qc_even.size()} gates")
print(f"Odd circuit:  {qc_odd.depth()} depth, {qc_odd.size()} gates")

## Simulator Baseline (reproduce the result)

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error


def heron_like_noise(depol_rate=0.01):
    model = NoiseModel()
    sq = thermal_relaxation_error(50.0, 30.0, 0.1)
    model.add_all_qubit_quantum_error(sq, ["rx", "ry", "rz", "sx", "x"])
    tq = depolarizing_error(depol_rate, 2)
    model.add_all_qubit_quantum_error(tq, ["cx", "ecr"])
    return model


def classical_fidelity(ideal_sv, counts, n_qubits):
    probs_ideal = np.abs(ideal_sv) ** 2
    total = sum(counts.values())
    probs_meas = np.zeros(2**n_qubits)
    for bitstr, count in counts.items():
        probs_meas[int(bitstr, 2)] = count / total
    return float(np.sum(np.sqrt(probs_ideal * probs_meas + 1e-20))) ** 2


# Simulator run
nm = heron_like_noise(depol_rate=0.02)
sim = AerSimulator(noise_model=nm)
shots = 10000

F_even_sim = []
F_odd_sim = []
for seed in range(20):
    job_e = sim.run(qc_even, shots=shots, seed_simulator=seed)
    job_o = sim.run(qc_odd, shots=shots, seed_simulator=seed)
    F_even_sim.append(classical_fidelity(sv_even, job_e.result().get_counts(), N))
    F_odd_sim.append(classical_fidelity(sv_odd, job_o.result().get_counts(), N))

from scipy.stats import ttest_rel

t_sim, p_sim = ttest_rel(F_even_sim, F_odd_sim)

print("=== SIMULATOR BASELINE (Heron-like noise, depol=0.02) ===")
print(f"Even: {np.mean(F_even_sim):.4f} +/- {np.std(F_even_sim):.4f}")
print(f"Odd:  {np.mean(F_odd_sim):.4f} +/- {np.std(F_odd_sim):.4f}")
print(f"t={t_sim:.3f}, p={p_sim:.6f}")
print(f"Asymmetry: {'CONFIRMED' if p_sim < 0.05 else 'NOT CONFIRMED'}")

## IBM Hardware Execution

Uncomment and run when ready to use IBM Quantum budget (10 min / 28-day cycle).
This uses ~2 minutes of the budget.

In [ ]:
# IBM_RUN = False  # Set True when ready to spend hardware budget
IBM_RUN = False

if IBM_RUN:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

    service = QiskitRuntimeService(
        channel="ibm_cloud",
        instance="crn:v1:bluemix:public:quantum-computing:us-east:"
        "a/78db885720334fd19191b33a839d0c35:"
        "841cc36d-0afd-4f96-ada2-8c56e1c443a0::",
        token="REDACTED",  # Use env var in production
    )
    backend = service.least_busy(min_num_qubits=4, operational=True)
    print(f"Backend: {backend.name}")

    # Transpile for hardware
    qc_even_hw = transpile(qc_even, backend=backend, optimization_level=2)
    qc_odd_hw = transpile(qc_odd, backend=backend, optimization_level=2)
    print(f"Even: {qc_even_hw.depth()} depth after transpile")
    print(f"Odd:  {qc_odd_hw.depth()} depth after transpile")

    # Run
    sampler = SamplerV2(backend)
    hw_shots = 8192

    job_even_hw = sampler.run([qc_even_hw] * 10, shots=hw_shots)
    job_odd_hw = sampler.run([qc_odd_hw] * 10, shots=hw_shots)

    print(f"Even job: {job_even_hw.job_id()}")
    print(f"Odd job:  {job_odd_hw.job_id()}")
    print("Jobs submitted. Retrieve with job IDs when complete.")
else:
    print("IBM_RUN=False. Set True and provide token to run on hardware.")
    print("Estimated budget usage: ~2 minutes of 10-minute cycle.")

In [ ]:
# Save results
results = {
    "experiment": "DLA parity sector decoherence asymmetry",
    "n_qubits": N,
    "ground_parity": round(parity_ev, 4),
    "simulator": {
        "noise_model": "heron_like, depol=0.02",
        "n_seeds": 20,
        "shots": shots,
        "mean_F_even": round(float(np.mean(F_even_sim)), 4),
        "mean_F_odd": round(float(np.mean(F_odd_sim)), 4),
        "t_stat": round(float(t_sim), 4),
        "p_value": round(float(p_sim), 6),
        "significant": bool(p_sim < 0.05),
    },
    "hardware": "pending" if not IBM_RUN else "submitted",
}
print("--- JSON RESULTS ---")
print(json.dumps(results, indent=2))